In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats
import multiprocessing
import matplotlib.gridspec as gridspec
import sys

# Used colorpalette
black = '#000000'
lightblack = '#333333'
darkgray = '#666666'
mediumgray = '#999999'
lightgray = '#CCCCCC'

darkred = '#FF0000'
red = '#FF3333'
lightred = '#FF6666'
pink ='#FF9999'
salmon = '#FFCCCC'


print('####################################################################')
print('#### Calculate ratios for scatterplots ©Antoni Gralak_01.07.2025 ###')
print('####################################################################')
print('Setting env...')

this_path = os.getcwd()
sys.path.append(os.path.join(this_path, '..'))

import utils

num_cores = 1
#num_cores = 23
experiment_names = ['exp10']
                #['exp1', 'exp2', 'exp3', 'exp4', 'exp5',
                #'exp6', 'exp7', 'exp8', 'exp9', 'exp10',
                #'exp11', 'exp12', 'exp13', 'exp14', 'exp15',
                #'exp16', 'exp17', 'exp18', 'exp19', 'exp20',
                #'exp21', 'exp22', 'exp23']
kmers = [6] #[6,7,8,9]
to_be_analyzed = ['ZNF891_FL']

####################################################################
#### Calculate ratios for scatterplots ©Antoni Gralak_01.07.2025 ###
####################################################################
Setting env...


In [3]:
def process_experiment(experiment_name):

    data_path = '../output_joint_analysis/'
    
    for TF in to_be_analyzed:
        for kmer in kmers:
            #kmer_path = data_path + f'/01_kmer_analysis/{experiment_name}_{TF}_{kmer}mer_enrichment.csv'
            p_val_path = data_path + '02_fishers_exact_test/significant_kmers/'
            ratio_path = data_path + '03_calculate_ratios/'
            save_path = data_path + '03_calculate_ratios/with_slopes/'

            try:
                os.makedirs(save_path)
            except FileExistsError:
                pass

    
        
            
            p_val = pd.read_csv(p_val_path + f'{TF}_{kmer}mer_significant.csv')
            significant_kmer_count = len(p_val)
            alert = None
            
            ratios = pd.read_csv(ratio_path + f'{experiment_name}_{TF}_{kmer}mer_ratio.csv')

            #Prepare alert messages
            messages = []
            if alert:
                messages.append(alert)
            elif significant_kmer_count < 20:
                messages.append("Poor enrichment (<20 significant kmers)")


            if p_val is not None:
                # Calculate kmer slopes!
                pvals_m = p_val[p_val['mod'] == 'methl'][['kmer', 'p_adjust']].rename(columns={'p_adjust': 'pval_methl'})
                pvals_nm = p_val[p_val['mod'] == 'nonmethl'][['kmer', 'p_adjust']].rename(columns={'p_adjust': 'pval_nonmethl'})

                ratios = ratios.merge(pvals_m, left_on='index', right_on='kmer', how='left').drop(columns='kmer')
                ratios = ratios.merge(pvals_nm, left_on='index', right_on='kmer', how='left').drop(columns='kmer')

                
                significant_kmers = ratios[(ratios['pval_methl'] <= 0.05) | (ratios['pval_nonmethl'] <= 0.05)]['index']
                
                ratios['significant'] = ratios['index'].isin(significant_kmers)

                

                most_significant_kmers = ratios[ratios['significant'] ==  True].nsmallest(50, ['pval_methl', 'pval_nonmethl'])['index']
                ratios['fill_plot'] = ratios['index'].isin(most_significant_kmers)

                most_significant_kmers = ratios[ratios['significant'] ==  True].nsmallest(50, ['pval_methl', 'pval_nonmethl'])['index']
                ratios['use_for_linreg'] = ratios['index'].isin(most_significant_kmers)

            else:
                ratios['significant'] = False
                ratios['use_for_linreg'] = False
                ratios['fill_plot'] = False

            

            lin_reg = {}
            coords = {}
            if p_val is not None:
                for spec, dataframe in ratios.groupby(['use_for_linreg', 'CpG']):
                    if spec[0]:
                        if spec[1]:
                            key = 'with_CG'
                        else:
                            key = 'no_CG'
                        
                        dataframe_no_nan = dataframe.dropna(subset=['eluted_methl', 'eluted_nonmethl']) #drop nan otherwise no linreg possible
                        if not dataframe_no_nan.empty:
                            x_l = dataframe_no_nan.eluted_methl
                            y_l = dataframe_no_nan.eluted_nonmethl

                            if len(x_l) < 2 or len(y_l) < 2 or x_l.nunique() == 1 or y_l.nunique() == 1:
                                continue
                            else:        
                                slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(x_l, y_l)
                                lin_reg[key] = [slope, intercept, r_value, p_value, std_err]
                                x_vals = np.linspace(x_l.min(), x_l.max(), 100)
                                y_vals = slope * x_vals + intercept
                                coords[key] = {'x': x_vals, 'y': y_vals}
            

            # Add regression slopes if available
            for key in lin_reg:
                slope = lin_reg[key][0]
                messages.append(f"Slope ({key}): {slope:.2f}")

            ratios.to_csv(save_path + f'{TF}_{kmer}_ratios.csv')
            
            # PLOT THE SCATTERPLOT WITH FILLED KMERS AND SLOPES


            fig = plt.figure(figsize=(9, 5.5))
            gs = gridspec.GridSpec(1, 2, width_ratios=[3.5, 1.5])
            ax0 = fig.add_subplot(gs[0])  # scatter plot
            ax1 = fig.add_subplot(gs[1])  # text box
            #fig, ax = plt.subplots(1, 1, figsize=(5.5, 5.5))

            x = ratios['eluted_methl']
            y = ratios['eluted_nonmethl']
            
            edgecolors = ratios['CpG'].map({True: darkred, False: lightblack})
            
            
            facecolors = [
                edgecolors.iloc[i] if sig else 'None'
                for i, sig in enumerate(ratios['fill_plot'])
            ]
            
            ax0.scatter(x=x, y=y, facecolors=facecolors, edgecolors=edgecolors, rasterized = True)
            if 'with_CG' in coords:
                ax0.plot(coords['with_CG']['x'], coords['with_CG']['y'], color=darkred)
            if 'no_CG' in coords:
                ax0.plot(coords['no_CG']['x'], coords['no_CG']['y'], color=lightblack)

            ax0.grid(visible=False)

            # Remove the top and right spines
            ax0.spines['top'].set_visible(False)
            ax0.spines['right'].set_visible(False)

            lower_limit = None
            upper_limit = max(x.max(), y.max()) * 1.05
            # Extent the axis by 5 % of max value
            ax0.set_xlim(lower_limit, upper_limit)
            ax0.set_ylim(lower_limit, upper_limit) 

            ax0.set_aspect('equal', adjustable='box')
                
                
            ax0.set_xlabel('methylated DNA', fontfamily='sans-serif', fontsize=10, fontstyle='italic')
            ax0.set_ylabel('unmethylated DNA', fontfamily='sans-serif', fontsize=10, fontstyle='italic')
                
            ax0.set_title(f"{TF} {kmer} enrichment, normalized by input", fontsize=10)

            # Display the messages in ax1
            ax1.axis('off')
            for i, msg in enumerate(messages):
                ax1.text(0, 1 - 0.1*i, msg, fontsize=9, fontfamily='monospace', va='top')
            
            plt.tight_layout()
            plt.savefig(save_path + f'{TF}_{kmer}_scatterplot_with_slope.pdf', dpi=400, bbox_inches='tight')
            plt.savefig(save_path + f'{TF}_{kmer}_scatterplot_with_slope.png', dpi=400, bbox_inches='tight')
            plt.close()

In [4]:
if __name__ == '__main__':

    with multiprocessing.Pool(num_cores) as pool:
        pool.map(process_experiment, experiment_names)